# Deep Research Agent — Demo Notebook

This notebook demonstrates the Deep Research Agent backed by a PostgreSQL database.

## Prerequisites
1. Start PostgreSQL: `docker compose up -d`
2. Install dependencies: `pip install -r requirements.txt`
3. Copy `.env.example` to `.env` and fill in your API keys

In [ ]:
# Verify environment and dependencies
import subprocess, sys
result = subprocess.run([sys.executable, '-c', 'import langgraph, langchain_openai, psycopg'], capture_output=True)
if result.returncode != 0:
    print("Installing dependencies...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', '../requirements.txt'], check=True)
print('Dependencies OK')

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

import config
print(f'Model: {config.OPENAI_MODEL}')
print(f'DB: {config.DATABASE_URL}')
print(f'Tavily configured: {bool(config.TAVILY_API_KEY)}')

In [ ]:
# Test PostgreSQL connection
import psycopg

with psycopg.connect(config.DATABASE_URL) as conn:
    version = conn.execute('SELECT version()').fetchone()[0]
    print(f'Connected: {version[:50]}')

    # Enable pgvector extension
    conn.execute('CREATE EXTENSION IF NOT EXISTS vector')
    conn.commit()
    print('pgvector extension: OK')

## Run a Research Query

Each run is identified by a `thread_id`. Re-running with the same `thread_id` resumes from the last checkpoint.

In [ ]:
from agent import run_research
import uuid

TOPIC = "What are the latest advancements in quantum computing and their practical applications?"
THREAD_ID = str(uuid.uuid4())   # New thread each run; reuse an ID to resume

print(f'Thread ID: {THREAD_ID}')
print(f'Topic: {TOPIC}')
print('Running research agent...')

result = run_research(TOPIC, THREAD_ID)
print('Done!')

In [ ]:
# Display the research plan
print('## Research Plan')
for i, q in enumerate(result['research_plan'], 1):
    print(f'{i}. {q}')

In [ ]:
# Display sources gathered
print(f'## Sources ({len(result["sources"])} found)')
for s in result['sources']:
    print(f"- [{s['title']}]({s['url']}) — score: {s['relevance_score']:.2f}")

In [ ]:
# Display the final report
from IPython.display import Markdown, display
display(Markdown(result['final_report']))

## Inspect the PostgreSQL State

LangGraph stores all checkpoints in PostgreSQL. Let's look at what was persisted.

In [ ]:
import psycopg

with psycopg.connect(config.DATABASE_URL) as conn:
    tables = conn.execute(
        "SELECT table_name FROM information_schema.tables WHERE table_schema = 'public'"
    ).fetchall()
    print('Tables in database:')
    for (t,) in tables:
        count = conn.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
        print(f'  {t}: {count} rows')

In [ ]:
# Resume from a previous thread (paste a THREAD_ID from above)
# RESUME_THREAD_ID = "paste-thread-id-here"
# result_resumed = run_research(TOPIC, RESUME_THREAD_ID)  # Picks up from last checkpoint
print('Uncomment the lines above and paste a thread ID to resume a previous run')

## Store Documents in the Vector Store

Persist the gathered sources into PostgreSQL with pgvector for semantic retrieval.

In [ ]:
from storage import ResearchDocumentStore

store = ResearchDocumentStore(collection="research_docs")
store.add_sources(result['sources'], metadata={"topic": TOPIC, "thread_id": THREAD_ID})
print(f'Stored {len(result["sources"])} documents in PostgreSQL vector store')

In [ ]:
# Semantic search over stored research
similar_docs = store.similarity_search('quantum error correction', k=3)
print(f'Found {len(similar_docs)} similar documents:')
for doc in similar_docs:
    print(f"  - {doc.metadata.get('title', 'Untitled')}: {doc.page_content[:150]}...")